# Camera-aware adaptation: a better identity model without test labels

Every post-hoc fix failed because **CSCI's fingerprints are not consistent across cameras**. This notebook **improves the fingerprints themselves**, using only unlabeled footage from the deployment cameras.

- **Adapter:** a small residual network on top of the frozen fingerprints. It starts as the identity (no change), with an anchor loss so it stays close to the original.
- **Each round (5 rounds):** cluster all clips into pseudo-identities (no labels), then train with a **camera-aware proxy contrastive loss**. It pulls each pseudo-identity's clips together **across cameras** and pushes the hardest look-alike identities apart.
- **Pseudo-label setting:** reported at three values (0.5%, 1%, 2% of clip pairs) as a sensitivity check. The final row picks the setting using the **other** group's labels only (calibration people), and is evaluated on the held-out group.
- **Cell 4:** mAP and Rank-1, plus consistent ID at matched precision, original vs adapted.
- **Cell 5:** the 50-test robustness check (25 random person splits × 2 directions), with a paired comparison.

Test labels are **never used for adaptation**, only for evaluation. A CPU runtime works (a GPU is faster); about 15 to 25 minutes in total. Results go to `MEVID-results/CameraAdapt-SameClothes`.

In [ ]:
# 1. Load test fingerprints (CSCI, the OSNet baseline), queries and groups.
#    Labels are used ONLY for evaluation, never for adaptation.
import os, glob, json, time, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return os.path.dirname(h[0])
    raise FileNotFoundError(pats[0])

CS = find([f'{SD}/Data/MEVID-results/CSCI-SameClothes/'
           'test_templates_csci_sc.npz',
           f'{SD}/**/CSCI-SameClothes/test_templates_csci_sc.npz'])
TR = find([f'{SD}/Data/MEVID-results/Tracker-SameClothes/'
           'test_templates_tracker_sc.npz',
           f'{SD}/**/Tracker-SameClothes/test_templates_tracker_sc.npz'])
OUT = os.path.join(os.path.dirname(CS), 'CameraAdapt-SameClothes')
os.makedirs(OUT, exist_ok=True)


def unit(x):
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True),
                           1e-8)).astype(np.float32)


te = np.load(f'{CS}/test_templates_csci_sc.npz')
tte = np.load(f'{TR}/test_templates_tracker_sc.npz')
pid, cam, qidx, gidx = te['pid'], te['cam'], te['qidx'], te['gidx']
FEATS = {'CSCI': unit(te['templates']),
         'OSNet baseline (OSNet)': unit(tte['zs'])}
proto = json.load(open(f'{CS}/PROTOCOL_frozen_sc.json'))
fold = np.array([proto['folds'][str(int(p))] for p in pid])

# all settings fixed BEFORE any result; the pseudo-label setting is
# reported at three values (sensitivity), none chosen on test labels
SETTINGS = {'rounds': 5, 'epochs_per_round': 10, 'lr': 3e-4,
            'temperature': 0.1, 'inter_weight': 0.5, 'hard_neg': 50,
            'anchor_weight': 1.0,
            'hidden': 512, 'pseudo_pair_fractions': [0.005, 0.01, 0.02],
            'n_splits': 25, 'seed': 2027}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
print('Clips', len(pid), '| cameras', len(np.unique(cam)),
      '| results ->', OUT)


In [ ]:
# 2. Metrics (same as all earlier notebooks) + standard mAP / Rank-1
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.optimize import linear_sum_assignment


def _cont(pred, p):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1))
    np.add.at(M, (pi, ci), 1)
    return M, pi, ci


def score(pred, p, c):
    M, pi, ci = _cont(pred, p)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0)
    tp = (M * (M - 1)).sum()
    same_id = (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    K = np.zeros((M.shape[0], ki.max() + 1, M.shape[1]))
    np.add.at(K, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (K.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (K ** 2).sum((1, 2))).sum()
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'pair precision %': (100 * tp / same_id if same_id
                                 else np.nan),
            'cross-camera recall %': 100 * x_same / max(1, x_tot)}


GRID = np.round(np.arange(0.01, 0.951, 0.01), 2)


def sweep(Z, p, c):
    D = np.clip(1 - Z @ Z.T, 0, None)
    D = (D + D.T) / 2
    L = linkage(D[np.triu_indices(len(D), 1)], 'average')
    return {d: score(fcluster(L, d, 'distance'), p, c) for d in GRID}


def map_rank1(Z):
    S = Z[qidx] @ Z[gidx].T
    aps, r1 = [], []
    for i, q in enumerate(qidx):
        sp = np.flatnonzero(pid[gidx] == pid[q])
        sc = np.flatnonzero(cam[gidx] == cam[q])
        good, junk = np.setdiff1d(sp, sc), np.intersect1d(sp, sc)
        if not good.size:
            continue
        o = np.argsort(-S[i]); o = o[~np.isin(o, junk)]
        hits = np.isin(o, good); r = np.flatnonzero(hits)
        aps.append(np.mean((np.arange(len(r)) + 1) / (r + 1)))
        r1.append(float(hits[0]))
    return 100 * np.mean(aps), 100 * np.mean(r1)


def at_precision(sw, lvl):
    ok = [v for v in sw.values() if v['pair precision %'] >= lvl]
    return max(ok, key=lambda v: v['IDF1 %']) if ok else None


In [ ]:
# 3. Camera-aware adaptation (unsupervised, no labels):
#    a small residual adapter on the frozen fingerprints, starting as the
#    identity. Each round: cluster all clips into pseudo-identities, then
#    train with a camera-aware proxy contrastive loss that pulls each
#    pseudo-identity's clips TOGETHER ACROSS CAMERAS.
import torch, torch.nn as nn, torch.nn.functional as F
dev = 'cuda' if torch.cuda.is_available() else 'cpu'


class Adapter(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Linear(h, d))
        nn.init.zeros_(self.f[2].weight); nn.init.zeros_(self.f[2].bias)

    def forward(self, x):
        return F.normalize(x + self.f(x), dim=1)


def pseudo_labels(Z, frac, cut=None):
    """average-linkage clusters as pseudo-identities. In round 1 the cut
    is chosen so that about `frac` of all clip pairs share a cluster (a
    crowd-size prior, no labels); later rounds reuse that cut, so merging
    cannot run away. Clusters of one clip are dropped (label -1)."""
    D = np.clip(1 - Z @ Z.T, 0, None); D = (D + D.T) / 2
    L = linkage(D[np.triu_indices(len(D), 1)], 'average')
    n = len(Z)
    if cut is None:
        cut = 0.01
        for d in np.arange(0.01, 0.99, 0.01):
            _, cnt = np.unique(fcluster(L, d, 'distance'),
                               return_counts=True)
            if (cnt * (cnt - 1)).sum() / (n * (n - 1)) > frac:
                break
            cut = d
    lab = fcluster(L, cut, 'distance')
    _, inv, cnt = np.unique(lab, return_inverse=True, return_counts=True)
    return np.where(cnt[inv] > 1, inv, -1), cut


def adapt(X, c, frac):
    st = SETTINGS
    torch.manual_seed(st['seed']); np.random.seed(st['seed'])
    x = torch.tensor(X, device=dev)
    ci = torch.tensor(np.unique(c, return_inverse=True)[1], device=dev)
    model = Adapter(X.shape[1], st['hidden']).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=st['lr'])
    T, cut = st['temperature'], None
    for rnd in range(st['rounds']):
        with torch.no_grad():
            Z = model(x).cpu().numpy()
        y, cut = pseudo_labels(Z, frac, cut)
        keep = np.flatnonzero(y >= 0)
        yk = torch.tensor(y[keep], device=dev)
        ck = ci[keep]
        # camera-aware proxies: one per (pseudo-identity, camera)
        key = yk * (int(ci.max()) + 1) + ck
        ukey, kinv = torch.unique(key, return_inverse=True)
        p_id = ukey // (int(ci.max()) + 1)
        p_cam = ukey % (int(ci.max()) + 1)
        for ep in range(st['epochs_per_round']):
            z = model(x[keep])
            with torch.no_grad():
                P = torch.zeros(len(ukey), z.shape[1], device=dev)
                P.index_add_(0, kinv, z)
                P = F.normalize(P, dim=1)
            logits = z @ P.T / T
            # intra-camera: own proxy vs other proxies of the same camera
            same_cam = p_cam[None, :] == ck[:, None]
            intra = logits.masked_fill(~same_cam, -1e9)
            l_intra = F.cross_entropy(intra, kinv)
            # inter-camera: proxies of the same pseudo-identity on OTHER
            # cameras vs the hardest other-identity proxies
            pos = (p_id[None, :] == yk[:, None]) & ~same_cam
            neg_logits = logits.masked_fill(p_id[None, :] == yk[:, None],
                                            -1e9)
            hn = neg_logits.topk(min(st['hard_neg'], neg_logits.shape[1]),
                                 dim=1).values
            has = pos.any(1)
            if has.any():
                lp = logits[has]; ps = pos[has]; h_ = hn[has]
                den = torch.logsumexp(torch.cat([lp.unsqueeze(2),
                                                 h_.unsqueeze(1).expand(
                                                     -1, lp.shape[1], -1)],
                                                2), dim=2)
                l_inter = -((lp - den) * ps).sum(1) / ps.sum(1)
                l_inter = l_inter.mean()
            else:
                l_inter = torch.zeros((), device=dev)
            # anchor: stay close to the original fingerprints (no collapse)
            l_anchor = (1 - (z * x[keep]).sum(1)).mean()
            loss = (l_intra + st['inter_weight'] * l_inter
                    + st['anchor_weight'] * l_anchor)
            opt.zero_grad(); loss.backward(); opt.step()
        print(f'    round {rnd + 1}: {len(np.unique(y[keep]))} '
              f'pseudo-identities, {len(keep)} clips, loss {loss.item():.3f}')
    with torch.no_grad():
        return model(x).cpu().numpy()


ADAPTED = {}
for bb, X in FEATS.items():
    for frac in SETTINGS['pseudo_pair_fractions']:
        t0 = time.time()
        print(f'{bb}, pseudo-pair fraction {frac}:')
        ADAPTED[(bb, frac)] = adapt(X, cam, frac)
        np.save(f"{OUT}/adapted_{bb.split()[0]}_{frac}.npy",
                ADAPTED[(bb, frac)])
        print(f'  done in {time.time() - t0:.0f} s')


In [ ]:
# 4. RESULTS vs the frozen backbone: retrieval accuracy, and consistent
#    ID at matched precision (mean of groups A and B)
rows = []
for bb, X in FEATS.items():
    variants = [('original', X)] + [(f'adapted (frac {f})',
                                     ADAPTED[(bb, f)])
                                    for f in SETTINGS['pseudo_pair_fractions']]
    for name, Z in variants:
        m, r1 = map_rank1(Z)
        row = {'backbone': bb, 'features': name, 'mAP %': m, 'Rank-1 %': r1}
        for lvl in [90, 95]:
            vals = []
            for g in 'AB':
                ig = np.flatnonzero(fold == g)
                b = at_precision(sweep(Z[ig], pid[ig], cam[ig]), lvl)
                if b:
                    vals.append(b)
            if vals:
                row[f'IDF1 @{lvl}%'] = np.mean([v['IDF1 %'] for v in vals])
                row[f'IDs/person @{lvl}%'] = np.mean(
                    [v['IDs per person'] for v in vals])
                row[f'cross-cam recall @{lvl}%'] = np.mean(
                    [v['cross-camera recall %'] for v in vals])
        rows.append(row)
# honest selection of the pseudo-label setting: chosen on the OTHER
# group's labels (calibration people), evaluated on the held-out group
for bb in FEATS:
    for lvl in [90, 95]:
        vals = []
        for cal, ev in [('A', 'B'), ('B', 'A')]:
            ic, ie = np.flatnonzero(fold == cal), np.flatnonzero(fold == ev)
            best, bf = -1, None
            for f in SETTINGS['pseudo_pair_fractions']:
                Z = ADAPTED[(bb, f)]
                b = at_precision(sweep(Z[ic], pid[ic], cam[ic]), lvl)
                if b and b['IDF1 %'] > best:
                    best, bf = b['IDF1 %'], f
            if bf is not None:
                Z = ADAPTED[(bb, bf)]
                b = at_precision(sweep(Z[ie], pid[ie], cam[ie]), lvl)
                if b:
                    vals.append(b)
        if vals:
            r = next((r for r in rows if r['backbone'] == bb and
                      r['features'] == 'adapted (setting chosen on '
                                       'other group)'), None)
            if r is None:
                r = {'backbone': bb, 'features':
                     'adapted (setting chosen on other group)'}
                rows.append(r)
            r[f'IDF1 @{lvl}%'] = np.mean([v['IDF1 %'] for v in vals])
            r[f'IDs/person @{lvl}%'] = np.mean([v['IDs per person']
                                                for v in vals])
            r[f'cross-cam recall @{lvl}%'] = np.mean(
                [v['cross-camera recall %'] for v in vals])
RES = pd.DataFrame(rows).round(2)
RES.to_csv(f'{OUT}/adaptation_results.csv', index=False)
print('BACKBONE vs CAMERA-AWARE ADAPTATION (no test labels used for '
      'adaptation; the last row picks the setting on the other group)')
display(RES)


In [ ]:
# 5. ROBUSTNESS: 25 random person splits x 2 directions; point
#    threshold set on the calibration people (pair precision >= 95%),
#    applied to the test people. Original vs each adapted version.
rng = np.random.default_rng(SETTINGS['seed'])
people = np.unique(pid)
splits = [set(rng.choice(people, len(people) // 2, replace=False))
          for _ in range(SETTINGS['n_splits'])]
rows = []
for bb, X in FEATS.items():
    variants = [('original', X)] + [(f'adapted (frac {f})',
                                     ADAPTED[(bb, f)])
                                    for f in SETTINGS['pseudo_pair_fractions']]
    for name, Z in variants:
        for k, a in enumerate(splits):
            grp = np.array(['A' if q in a else 'B' for q in pid])
            sw = {}
            for g in 'AB':
                ig = np.flatnonzero(grp == g)
                sw[g] = sweep(Z[ig], pid[ig], cam[ig])
            for cal, ev in [('A', 'B'), ('B', 'A')]:
                d = None
                for x in GRID:
                    pp = sw[cal][x]['pair precision %']
                    if pp == pp and pp >= 95:
                        d = x
                if d is None:
                    continue
                e = sw[ev][d]
                rows.append({'backbone': bb, 'features': name, 'split': k,
                             'direction': f'{cal}->{ev}',
                             'promised %': sw[cal][d]['pair precision %'],
                             'delivered %': e['pair precision %'],
                             'IDF1 %': e['IDF1 %'],
                             'IDs per person': e['IDs per person']})
    print(f'{bb}: done')
REP = pd.DataFrame(rows)
REP['broken (<93%)'] = (REP['delivered %'] < 93).astype(float)
summ = REP.groupby(['backbone', 'features'], sort=False)[
    ['promised %', 'delivered %', 'broken (<93%)', 'IDF1 %',
     'IDs per person']].mean().round(2)
summ.to_csv(f'{OUT}/adaptation_robustness.csv')
print('POINT THRESHOLD over 50 tests (held-out people)')
display(summ)

key = ['backbone', 'split', 'direction']
base = REP[REP.features == 'original'].set_index(key)
out = []
for name in REP.features.unique():
    if name == 'original':
        continue
    j = base.join(REP[REP.features == name].set_index(key),
                  lsuffix='_o', rsuffix='_a', how='inner')
    for bb, jj in j.groupby(level=0):
        out.append({'backbone': bb, 'features': name, 'tests': len(jj),
                    'higher IDF1 (% of tests)':
                        round(100 * (jj['IDF1 %_a'] > jj['IDF1 %_o']).mean(),
                              1),
                    'mean IDF1 change': round((jj['IDF1 %_a']
                                               - jj['IDF1 %_o']).mean(), 2),
                    'fewer IDs/person (% of tests)':
                        round(100 * (jj['IDs per person_a']
                                     < jj['IDs per person_o']).mean(), 1)})
paired = pd.DataFrame(out)
paired.to_csv(f'{OUT}/adaptation_paired.csv', index=False)
print('\nPaired with the original backbone (same split and direction):')
display(paired)
